In [0]:
from scipy.stats import ks_2samp

def tool_rigorous_validation(model_results, data_dict, min_auc_threshold=0.70):
    """Aplica Quality Gates baseados em métricas de crédito (AUC e KS)."""
    model = model_results["model"]
    X_test, y_test = data_dict["X_test"], data_dict["y_test"]
    
    preds_proba = model.predict_proba(X_test)[:, 1]
    
    # Cálculo do AUC
    auc_score = roc_auc_score(y_test, preds_proba)
    
    # Cálculo do KS (Kolmogorov-Smirnov)
    df_eval = pd.DataFrame({"prob": preds_proba, "target": y_test})
    class_0 = df_eval[df_eval["target"] == 0]["prob"]
    class_1 = df_eval[df_eval["target"] == 1]["prob"]
    ks_stat, _ = ks_2samp(class_0, class_1)
    
    print(f"[Agente] Validação - AUC: {auc_score:.4f} | KS: {ks_stat:.4f}")
    
    if auc_score >= min_auc_threshold:
        print("[Agente] Status: APROVADO pelo Quality Gate.")
        return {"approved": True, "auc": auc_score, "ks": ks_stat}
    else:
        print("[Agente] Status: REJEITADO. O modelo não atingiu o limiar mínimo.")
        return {"approved": False, "auc": auc_score, "ks": ks_stat}